In [19]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import statsmodels.api as sm
from scipy import stats
from scipy.optimize import minimize
import warnings
warnings.filterwarnings('ignore')

# ===================== 全局设置 =====================
plt.rcParams['font.sans-serif'] = ['SimHei']
plt.rcParams['axes.unicode_minus'] = False
FACTOR_NAME = 'momentum_factor'
RETURN_NAME = 'future_return'
N_LAYERS = 5
STANDARDIZE_METHODS = ['global_zscore', 'industry_zscore']
INDEX_DATE = 'DATE'
INDEX_CODE = 'CODE'
# 数据路径
DAILY_DATA_PATH = r'C:\Users\28405\Desktop\【2】实习笔试(笔试答案用本人姓名命名文件)\basic_data\daily_data.parquet'
INDUSTRY_PATH = r'C:\Users\28405\Desktop\【2】实习笔试(笔试答案用本人姓名命名文件)\basic_data\industry.parquet'
ST_PATH = r'C:\Users\28405\Desktop\【2】实习笔试(笔试答案用本人姓名命名文件)\basic_data\st.parquet'
SUSPEND_PATH = r'C:\Users\28405\Desktop\【2】实习笔试(笔试答案用本人姓名命名文件)\basic_data\停牌.parquet'

# ===================== 步骤1：载入数据 =====================
print("=== 步骤1：载入数据 ===")

# 1.1 载入daily_data
daily_value_cols = ['DY-ADJ_AF-OPEN_PRICE_2', 'DY-ADJ_AF-CLOSE_PRICE_2', 'DY-BASIC-MARKET_VALUE']
daily_df = pd.read_parquet(DAILY_DATA_PATH, columns=daily_value_cols)
daily_df = daily_df.reset_index()
daily_df.rename(columns={
    INDEX_DATE: 'date',
    INDEX_CODE: 'code',
    'DY-ADJ_AF-OPEN_PRICE_2': 'open',
    'DY-ADJ_AF-CLOSE_PRICE_2': 'close',
    'DY-BASIC-MARKET_VALUE': 'market_value'
}, inplace=True)
# 确保market_value是数值型，剔除非数值值
daily_df['market_value'] = pd.to_numeric(daily_df['market_value'], errors='coerce')
daily_df['date'] = pd.to_datetime(daily_df['date'])
daily_df.set_index(['date', 'code'], inplace=True)
daily_df = daily_df.dropna()
print(f"daily_data处理后形状：{daily_df.shape}")

# 1.2 载入行业数据
industry_df = pd.read_parquet(INDUSTRY_PATH)
industry_df = industry_df.reset_index()
industry_df.rename(columns={INDEX_DATE: 'date', INDEX_CODE: 'code'}, inplace=True)
industry_df['date'] = pd.to_datetime(industry_df['date'])
industry_df = industry_df[['date', 'code', 'LEVEL1_NAME']].set_index(['date', 'code'])
print(f"行业数据处理后形状：{industry_df.shape}")

# 1.3 载入ST数据
st_df = pd.read_parquet(ST_PATH)
st_df.index.name = INDEX_DATE
st_df = st_df.reset_index()
st_df.rename(columns={INDEX_DATE: 'date'}, inplace=True)
st_long = pd.melt(st_df, id_vars=['date'], var_name='code', value_name='is_st')
st_long['date'] = pd.to_datetime(st_long['date'])
st_long['is_st'] = st_long['is_st'].notna().astype(int)
st_long.set_index(['date', 'code'], inplace=True)
print(f"ST数据处理后形状：{st_long.shape}")

# 1.4 载入停牌数据
suspend_df = pd.read_parquet(SUSPEND_PATH)
suspend_df.rename(columns={'股票代码': 'code', '日期': 'date', '是否停牌': 'is_suspend'}, inplace=True)
suspend_df['date'] = pd.to_datetime(suspend_df['date'])
suspend_df.set_index(['date', 'code'], inplace=True)
print(f"停牌数据处理后形状：{suspend_df.shape}")

# 1.5 合并数据
base_df = daily_df.join(industry_df, how='left').join(st_long, how='left').join(suspend_df, how='left')
base_df['is_st'] = base_df['is_st'].fillna(0)
base_df['is_suspend'] = base_df['is_suspend'].fillna(0)
base_df = base_df.dropna()
print(f"合并后基础数据形状：{base_df.shape}")

# ===================== 步骤2：计算因子和收益率 =====================
print("=== 步骤2：计算因子和收益率 ===")

# 计算因子，确保结果是数值型
base_df[FACTOR_NAME] = base_df['close'] / base_df.groupby(level='code')['close'].shift(5) - 1
base_df[FACTOR_NAME] = pd.to_numeric(base_df[FACTOR_NAME], errors='coerce')

# 计算未来收益率
base_df['open_t1'] = base_df.groupby(level='code')['open'].shift(-1)
base_df['open_t2'] = base_df.groupby(level='code')['open'].shift(-2)
base_df[RETURN_NAME] = base_df['open_t2'] / base_df['open_t1'] - 1
base_df[RETURN_NAME] = pd.to_numeric(base_df[RETURN_NAME], errors='coerce')

# 剔除因子/收益率缺失的行
base_df = base_df.dropna(subset=[FACTOR_NAME, RETURN_NAME])
print(f"计算后数据形状：{base_df.shape}")

# ===================== 步骤3：数据筛选 =====================
print("=== 步骤3：数据筛选 ===")

base_df = base_df[base_df['is_st'] == 0]
base_df['is_suspend_t1'] = base_df.groupby(level='code')['is_suspend'].shift(-1)
base_df = base_df[base_df['is_suspend_t1'] == 0]
base_df['close_t1'] = base_df.groupby(level='code')['close'].shift(-1)
base_df['pct_chg_t1'] = (base_df['close_t1'] / base_df['open_t1']) - 1
base_df = base_df[(base_df['pct_chg_t1'].abs() < 0.097)]

# 剔除上市60天内股票
list_date = base_df.reset_index().groupby('code')['date'].min()
base_df = base_df.reset_index()
base_df['list_date'] = base_df['code'].map(list_date)
base_df['list_days'] = (base_df['date'] - base_df['list_date']).dt.days
base_df = base_df[base_df['list_days'] >= 60]
base_df.set_index(['date', 'code'], inplace=True)

print(f"筛选后最终数据形状：{base_df.shape}")

# ===================== 步骤4：数据整理 =====================
core_df = base_df[[FACTOR_NAME, RETURN_NAME, 'market_value', 'LEVEL1_NAME']].copy()
# 确保所有数值列是float类型
for col in [FACTOR_NAME, RETURN_NAME, 'market_value']:
    core_df[col] = core_df[col].astype(float)
print(f"核心数据字段：{core_df.columns.tolist()}")
print(f"核心数据类型：\n{core_df.dtypes}")

# ===================== 步骤5：因子标准化 =====================
print("=== 步骤5：因子标准化 ===")

# 方法1：全局Z-score
core_df['factor_std1'] = core_df.groupby(level='date')[FACTOR_NAME].transform(
    lambda x: (x - x.mean()) / x.std() if x.std() != 0 else 0
)

# 方法2：行业内Z-score
core_df['factor_std2'] = core_df.groupby(['date', 'LEVEL1_NAME'])[FACTOR_NAME].transform(
    lambda x: (x - x.mean()) / x.std() if x.std() != 0 else 0
)

# 剔除标准化后缺失/无穷大的值
core_df = core_df.replace([np.inf, -np.inf], np.nan)
core_df = core_df.dropna(subset=['factor_std1', 'factor_std2'])
print(f"标准化后数据形状：{core_df.shape}")

# ===================== 步骤6：因子中性化（核心修正） =====================
print("=== 步骤6：因子中性化 ===")

def neutralize_factor(df, factor_col):
    """修正：确保X和y均为数值型，剔除异常值"""
    neutralized = []
    for date, date_df in df.groupby(level='date'):
        # 1. 筛选有效数据：因子值、市值均为数值，且市值>0（避免log出错）
        valid_df = date_df[
            (pd.to_numeric(date_df[factor_col], errors='coerce').notna()) &
            (pd.to_numeric(date_df['market_value'], errors='coerce').notna()) &
            (date_df['market_value'] > 0)
        ].copy()
        
        if len(valid_df) < 30:  # 样本量过少跳过该截面
            # 填充NaN，后续会剔除
            neutralized.append(pd.Series(np.nan, index=date_df.index))
            continue
        
        # 2. 构建回归变量（确保均为数值型）
        X = pd.get_dummies(valid_df['LEVEL1_NAME'], prefix='ind', drop_first=True)  # drop_first避免多重共线性
        X['log_mv'] = np.log(valid_df['market_value'])
        X = sm.add_constant(X)
        # 确保X是float类型
        X = X.astype(float)
        
        # 3. 构建因变量（因子值）
        y = valid_df[factor_col].astype(float)
        
        # 4. 回归求残差
        try:
            model = sm.OLS(y, X).fit()
            res = model.resid
            # 对齐索引（将残差映射回原date_df）
            res_aligned = pd.Series(np.nan, index=date_df.index)
            res_aligned[valid_df.index] = res
            neutralized.append(res_aligned)
        except:
            # 回归失败时填充NaN
            neutralized.append(pd.Series(np.nan, index=date_df.index))
    
    return pd.concat(neutralized)

# 6a) 原始因子中性化
core_df['factor_neu_raw'] = neutralize_factor(core_df, FACTOR_NAME)

# 6b) 两种标准化因子中性化
core_df['factor_neu_std1'] = neutralize_factor(core_df, 'factor_std1')
core_df['factor_neu_std2'] = neutralize_factor(core_df, 'factor_std2')

# 剔除中性化后缺失的值
target_factors = ['factor_neu_std1', 'factor_neu_std2']
core_df = core_df.replace([np.inf, -np.inf], np.nan)
core_df = core_df.dropna(subset=target_factors + [RETURN_NAME])
print(f"中性化目标因子：{target_factors}")
print(f"中性化后数据形状：{core_df.shape}")

# ===================== 步骤7：计算IC值 =====================
print("=== 步骤7：计算IC值 ===")

def calc_ic(df, factor_col):
    return df.groupby(level='date').apply(
        lambda x: x[factor_col].corr(x[RETURN_NAME], method='spearman') if len(x) >= 30 else np.nan
    )

ic_dict = {f: calc_ic(core_df, f) for f in target_factors}
ic_df = pd.DataFrame(ic_dict).dropna()
print(f"IC计算完成，有效时间截面数：{len(ic_df)}")
print("IC均值：")
print(ic_df.mean().round(4))

# ===================== 步骤8：IC t检验 =====================
print("=== 步骤8：IC t检验 ===")

ic_stats = pd.DataFrame(index=target_factors, columns=['均值', '标准差', 't统计量', 'p值'])
for f in target_factors:
    ic_series = ic_df[f].dropna()
    mean_ic = ic_series.mean()
    std_ic = ic_series.std()
    t_stat = mean_ic / (std_ic / np.sqrt(len(ic_series))) if std_ic != 0 else 0
    p_val = 2 * (1 - stats.t.cdf(abs(t_stat), df=len(ic_series)-1)) if len(ic_series) > 1 else np.nan
    ic_stats.loc[f] = [mean_ic, std_ic, t_stat, p_val]

print("IC统计结果：")
print(ic_stats.round(4))

# ===================== 步骤9：分层回测 =====================
print("=== 步骤9：分层回测 ===")

def layer_backtest(df, factor_col):
    layer_returns = []
    for date, date_df in df.groupby(level='date'):
        if len(date_df) < 50:  # 样本量过少跳过
            continue
        date_df['rank'] = date_df[factor_col].rank(method='dense')
        # 确保分箱有效
        try:
            date_df['layer'] = pd.qcut(
                date_df['rank'], N_LAYERS, labels=range(1, N_LAYERS+1), duplicates='drop'
            )
            layer_ret = date_df.groupby('layer')[RETURN_NAME].mean()
            layer_returns.append(layer_ret)
        except:
            continue
    layer_ret_df = pd.DataFrame(layer_returns, index=ic_df.index[:len(layer_returns)])
    cum_ret = (1 + layer_ret_df).cumprod() - 1
    return cum_ret, layer_ret_df

backtest_res = {}
for f in target_factors:
    print(f"回测因子：{f}")
    cum_ret, layer_ret = layer_backtest(core_df, f)
    backtest_res[f] = (cum_ret, layer_ret)

# 回测指标
def calc_backtest_metrics(ret_df):
    if len(ret_df) == 0:
        return pd.DataFrame()
    annual_ret = (1 + ret_df.mean()) ** 252 - 1
    annual_vol = ret_df.std() * np.sqrt(252)
    sharpe = annual_ret / annual_vol if annual_vol != 0 else np.nan
    cum_ret = (1 + ret_df).cumprod() - 1
    max_dd = (cum_ret.cummax() - cum_ret).max() / (cum_ret.cummax() + 1e-8) if len(cum_ret) > 0 else np.nan
    return pd.DataFrame({
        '年化收益': annual_ret,
        '年化波动率': annual_vol,
        '夏普比率': sharpe,
        '最大回撤': max_dd
    })

first_f = target_factors[0]
cum_ret_first, layer_ret_first = backtest_res[first_f]
metrics_first = calc_backtest_metrics(layer_ret_first)
print(f"\n{first_f} 回测指标：")
print(metrics_first.round(4))

# ===================== 步骤10：绘制图表 =====================
print("=== 步骤10：绘制图表 ===")

for f in target_factors:
    cum_ret, layer_ret = backtest_res[f]
    ic_cum = ic_df[f].cumsum()
    
    fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))
    
    # 累计IC曲线
    ax1.plot(ic_cum.index, ic_cum.values, color='#2E86AB', linewidth=2, label='累计IC')
    ax1.set_title(f'{f} - 累计IC曲线', fontsize=14, fontweight='bold')
    ax1.set_xlabel('日期', fontsize=12)
    ax1.set_ylabel('累计IC值', fontsize=12)
    ax1.legend()
    ax1.grid(True, alpha=0.3)
    
    # 分层收益曲线
    colors = ['#A23B72', '#F18F01', '#C73E1D', '#2E86AB', '#F24236']
    for i, layer in enumerate(cum_ret.columns):
        ax2.plot(cum_ret.index, cum_ret[layer], color=colors[i], linewidth=2, label=f'第{layer}层')
    ax2.set_title(f'{f} - 分层回测累计收益曲线（等权）', fontsize=14, fontweight='bold')
    ax2.set_xlabel('日期', fontsize=12)
    ax2.set_ylabel('累计收益率', fontsize=12)
    ax2.legend()
    ax2.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f'{f}_累计IC和分层回测图.png', dpi=300, bbox_inches='tight')
    plt.show()
    print(f"图表保存：{f}_累计IC和分层回测图.png")

# ===================== 加分题：二次规划 =====================
def quadratic_programming_opt(df, factor_col):
    last_date = df.index.get_level_values('date').max()
    date_df = df.loc[last_date]
    target_stocks = date_df.nsmallest(100, factor_col).copy()
    
    if len(target_stocks) < 100:
        print(f"警告：{last_date}仅{len(target_stocks)}只股票，不足100只")
        return None
    
    codes = target_stocks.index.get_level_values('code').tolist()
    hist_returns = []
    for code in codes:
        hist = df.loc[(slice(None), code), RETURN_NAME].dropna().tail(20)
        hist_returns.append(hist.values)
    hist_returns = np.array(hist_returns).T
    cov_mat = np.cov(hist_returns) if hist_returns.shape[1] >= 2 else np.eye(len(codes))
    
    def obj(w):
        return 0.5 * np.dot(w.T, np.dot(cov_mat, w))
    
    constraints = [{'type': 'eq', 'fun': lambda w: np.sum(w) - 1}]
    for ind in target_stocks['LEVEL1_NAME'].unique():
        ind_mask = target_stocks['LEVEL1_NAME'] == ind
        constraints.append({'type': 'ineq', 'fun': lambda w, m=ind_mask: 0.2 - np.sum(w[m])})
    
    bounds = [(0, 0.05) for _ in range(len(target_stocks))]
    w0 = np.ones(len(target_stocks)) / len(target_stocks)
    
    res = minimize(obj, w0, method='SLSQP', constraints=constraints, bounds=bounds)
    target_stocks['组合权重'] = res.x.round(4)
    return target_stocks[['LEVEL1_NAME', RETURN_NAME, 'market_value', '组合权重']]

qp_res = quadratic_programming_opt(core_df, first_f)
if qp_res is not None:
    qp_res.to_csv(f'{first_f}_二次规划结果.csv', encoding='utf-8-sig')
    print(f"二次规划结果保存：{first_f}_二次规划结果.csv")

# ===================== 保存结果 =====================
ic_stats.to_csv('IC统计结果.csv', encoding='utf-8-sig')
metrics_first.to_csv(f'{first_f}_回测指标.csv', encoding='utf-8-sig')

print("\n✅ 所有步骤执行完成！")
print(f"提交文件清单：")
print(f"1. 代码文件：本人姓名_单因子检验.py")
print(f"2. 图表文件：{target_factors[0]}_累计IC和分层回测图.png、{target_factors[1]}_累计IC和分层回测图.png")

# ===================== 错误修正总结 =====================
print("\n🔧 本次错误修正：")
print(f"1. ValueError根源：OLS回归时X（行业虚拟变量+对数市值）或y（因子值）含object类型/异常值（inf/nan）")
print(f"2. 核心修正：")
print(f"   - 强制转换数值列为float，剔除非数值值")
print(f"   - 市值取log前确保>0，避免inf")
print(f"   - 行业虚拟变量添加drop_first=True，避免多重共线性")
print(f"   - 样本量过少（<30）的时间截面跳过，避免回归不稳定")
print(f"   - 捕获回归异常，填充NaN后统一剔除")

=== 步骤1：载入数据 ===
daily_data处理后形状：(8717164, 3)
行业数据处理后形状：(9012858, 1)
ST数据处理后形状：(16963830, 1)
停牌数据处理后形状：(8730456, 1)
合并后基础数据形状：(8717164, 6)
=== 步骤2：计算因子和收益率 ===
计算后数据形状：(8306610, 10)
=== 步骤3：数据筛选 ===
筛选后最终数据形状：(7719828, 15)
核心数据字段：['momentum_factor', 'future_return', 'market_value', 'LEVEL1_NAME']
核心数据类型：
momentum_factor    float64
future_return      float64
market_value       float64
LEVEL1_NAME         object
dtype: object
=== 步骤5：因子标准化 ===
标准化后数据形状：(7719821, 6)
=== 步骤6：因子中性化 ===
中性化目标因子：['factor_neu_std1', 'factor_neu_std2']
中性化后数据形状：(7719821, 9)
=== 步骤7：计算IC值 ===
IC计算完成，有效时间截面数：2869
IC均值：
factor_neu_std1   -0.0535
factor_neu_std2   -0.0531
dtype: float64
=== 步骤8：IC t检验 ===
IC统计结果：
                       均值       标准差       t统计量   p值
factor_neu_std1  -0.05352  0.106051 -27.031487  0.0
factor_neu_std2 -0.053082  0.106245 -26.761113  0.0
=== 步骤9：分层回测 ===
回测因子：factor_neu_std1
回测因子：factor_neu_std2


ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().